In [2]:
class Term:
    """Represents a variable, constant, or compound expression/predicate."""
    def __init__(self, name, args=None, is_var=False):
        self.name = name          # Symbol name, e.g., 'x', 'Riya', 'Eats'
        self.args = args or []    # List of argument Terms if predicate/compound
        self.is_var = is_var      # True if variable, False if constant/predicate

    def is_constant(self):
        return not self.is_var and len(self.args) == 0

    def is_compound(self):
        return len(self.args) > 0

    def __repr__(self):
        if self.is_compound():
            args_str = ", ".join(repr(a) for a in self.args)
            return f"{self.name}({args_str})"
        return self.name


def occurs_check(var, expr):
    """Checks if variable 'var' occurs within expression 'expr'."""
    if var.name == expr.name and var.is_var and expr.is_var:
        return True
    if expr.is_compound():
        return any(occurs_check(var, arg) for arg in expr.args)
    return False


def apply_subst(subst, term):
    """Applies substitution dictionary to a term recursively."""
    if term.is_var and term.name in subst:
        return apply_subst(subst, subst[term.name])
    if term.is_compound():
        new_args = [apply_subst(subst, arg) for arg in term.args]
        return Term(term.name, new_args, is_var=False)
    return term


def unify(psi1, psi2, SUBST=None):
    """
    Unification Algorithm (Unify(Ψ₁, Ψ₂)):
    Returns substitution dictionary on success, or 'FAILURE' on failure.
    """
    if SUBST is None:
        SUBST = {}

    # Step 1: If Ψ₁ or Ψ₂ is a variable or constant
    if (psi1.is_var or psi1.is_constant()) and (psi2.is_var or psi2.is_constant()):
        # Step 1a: If identical, return NIL (empty substitution)
        if psi1.name == psi2.name and psi1.is_var == psi2.is_var:
            return {}

    # Step 1b: If Ψ₁ is a variable
    if psi1.is_var:
        if occurs_check(psi1, psi2):
            return "FAILURE"
        return {psi1.name: psi2}

    # Step 1c: If Ψ₂ is a variable
    if psi2.is_var:
        if occurs_check(psi2, psi1):
            return "FAILURE"
        return {psi2.name: psi1}

    # Step 2: Check if initial predicate symbols are equal
    if psi1.name != psi2.name:
        return "FAILURE"

    # Step 3: Check if number of arguments match
    if len(psi1.args) != len(psi2.args):
        return "FAILURE"

    # Step 4: Set Substitution set (SUBST) to NIL (already initialized)
    # Step 5: For i = 1 to number of elements
    for i in range(len(psi1.args)):
        arg1 = apply_subst(SUBST, psi1.args[i])
        arg2 = apply_subst(SUBST, psi2.args[i])

        # Step 5a: Call Unify function on corresponding arguments
        S = unify(arg1, arg2)

        # Step 5b: If S = failure then return FAILURE
        if S == "FAILURE":
            return "FAILURE"

        # Step 5c: If S != NIL, update SUBST
        if S:
            # Apply S to existing bindings in SUBST
            for key in list(SUBST.keys()):
                SUBST[key] = apply_subst(S, SUBST[key])
            SUBST.update(S)

    # Step 6: Return SUBST
    return SUBST


def print_result(label, expr1, expr2, result):
    print(f"--- {label} ---")
    print(f"Ψ₁: {expr1}")
    print(f"Ψ₂: {expr2}")
    if result == "FAILURE":
        print("Result: No Unifier (FAILURE)\n")
    else:
        print("Result: Unifier Found!")
        bindings = [f"{k} / {v}" for k, v in result.items()]
        print(f"Substitution Set: {{ {', '.join(bindings)} }}\n")


# ==========================================
# TEST CASES
# ==========================================

# Case 1: Unifier Found (Success)
# Ψ₁ = Eats(x, Apple)
# Ψ₂ = Eats(Riya, y)
x = Term("x", is_var=True)
y = Term("y", is_var=True)
riya = Term("Riya")
apple = Term("Apple")

psi1_success = Term("Eats", [x, apple])
psi2_success = Term("Eats", [riya, y])

res_success = unify(psi1_success, psi2_success)
print_result("Test 1: Successful Unification", psi1_success, psi2_success, res_success)


# Case 2: No Unifier (Predicate Mismatch Failure)
# Ψ₁ = Eats(x, Apple)
# Ψ₂ = Drinks(Riya, y)
psi1_fail1 = Term("Eats", [x, apple])
psi2_fail1 = Term("Drinks", [riya, y])

res_fail1 = unify(psi1_fail1, psi2_fail1)
print_result("Test 2: Failure (Predicate Mismatch)", psi1_fail1, psi2_fail1, res_fail1)


# Case 3: No Unifier (Occurs Check Failure)
# Ψ₁ = Knows(x, x)
# Ψ₂ = Knows(y, Father(y)) -- trying to unify variable with expression containing itself
psi1_fail2 = Term("Knows", [x])
psi2_fail2 = Term("Knows", [Term("Father", [x])])

res_fail2 = unify(psi1_fail2, psi2_fail2)
print_result("Test 3: Failure (Occurs Check)", psi1_fail2, psi2_fail2, res_fail2)

--- Test 1: Successful Unification ---
Ψ₁: Eats(x, Apple)
Ψ₂: Eats(Riya, y)
Result: Unifier Found!
Substitution Set: { x / Riya, y / Apple }

--- Test 2: Failure (Predicate Mismatch) ---
Ψ₁: Eats(x, Apple)
Ψ₂: Drinks(Riya, y)
Result: No Unifier (FAILURE)

--- Test 3: Failure (Occurs Check) ---
Ψ₁: Knows(x)
Ψ₂: Knows(Father(x))
Result: No Unifier (FAILURE)

